# GitHub Rate Limits Explorer

Understanding REST API rate limits for bulk data ingestion from GitHub.

## Your Current Limits

**Authentication:** Personal Access Token (PAT)

### Primary Rate Limits
- **5,000 requests per hour** (authenticated PAT)
- Resets hourly
- Shared across ALL API calls

### Secondary Rate Limits
- **Max 100 concurrent requests** (REST + GraphQL combined)
- **Max 900 points per minute** on REST API
- Each GET request = 1 point
- Each POST/PATCH/PUT/DELETE = 5 points
- **Max 90 seconds CPU time per 60 seconds** of real time

In [1]:
import pandas as pd

# Rate limit constants
PRIMARY_LIMIT = 5000  # requests per hour
SECONDARY_LIMIT_PER_MINUTE = 900  # points per minute
MAX_CONCURRENT = 100

# Request costs (GET = 1 point, POST/PATCH/PUT/DELETE = 5 points)
REQUEST_COSTS = {
    'GET': 1,
    'POST': 5,
    'PATCH': 5,
    'PUT': 5,
    'DELETE': 5,
}

print(f"Primary Rate Limit: {PRIMARY_LIMIT} requests/hour")
print(f"Secondary Rate Limit: {SECONDARY_LIMIT_PER_MINUTE} points/minute")
print(f"Max Concurrent Requests: {MAX_CONCURRENT}")
print(f"\nAverage requests per second: {PRIMARY_LIMIT / 3600:.1f} req/sec")
print(f"Average requests per minute: {PRIMARY_LIMIT / 60:.0f} req/min")

Primary Rate Limit: 5000 requests/hour
Secondary Rate Limit: 900 points/minute
Max Concurrent Requests: 100

Average requests per second: 1.4 req/sec
Average requests per minute: 83 req/min


## Ingest Pipeline Costs

For ingesting issues, our pipeline makes 3 API calls per issue:

In [2]:
# Typical issue ingestion flow
operations = {
    'Fetch Issues List': {'calls_per_issue': 0, 'per_page': 100, 'description': '(paginated, ~1 call per 100 issues)'},
    'Fetch Comments': {'calls_per_issue': 1, 'description': '(1 call per issue, regardless of comment count)'},
    'Fetch Events': {'calls_per_issue': 1, 'description': '(1 call per issue, regardless of event count)'},
}

# Total per issue (excluding pagination for issue list)
calls_per_issue = 1 + 1  # comments + events
points_per_issue = calls_per_issue * REQUEST_COSTS['GET']  # All GET requests = 1 point each

print(f"API Calls per Issue: {calls_per_issue}")
print(f"Points per Issue: {points_per_issue}\n")

# Show breakdown
for op, details in operations.items():
    print(f"{op}: {details['description']}")

API Calls per Issue: 2
Points per Issue: 2

Fetch Issues List: (paginated, ~1 call per 100 issues)
Fetch Comments: (1 call per issue, regardless of comment count)
Fetch Events: (1 call per issue, regardless of event count)


## Rate Limit Calculator

In [3]:
def calculate_ingest_limits(num_issues):
    """
    Calculate primary and secondary rate limit impact for ingesting N issues.
    """
    calls_per_issue = 2  # comments + events
    points_per_issue = calls_per_issue * 1  # GET requests = 1 point
    
    # Primary limit impact
    total_calls = num_issues * calls_per_issue
    primary_hours_needed = total_calls / PRIMARY_LIMIT
    
    # Secondary limit impact
    total_points = num_issues * points_per_issue
    minutes_for_secondary = total_points / SECONDARY_LIMIT_PER_MINUTE
    
    # If we had perfect pacing (no concurrency, sequential)
    seconds_needed_sequential = (total_points / SECONDARY_LIMIT_PER_MINUTE) * 60
    
    return {
        'num_issues': num_issues,
        'total_calls': total_calls,
        'total_points': total_points,
        'primary_hours': primary_hours_needed,
        'secondary_minutes': minutes_for_secondary,
        'sequential_seconds': seconds_needed_sequential,
    }

# Test with different repo sizes
test_sizes = [100, 269, 500, 1000, 2500, 5000, 10000]
results = []

for size in test_sizes:
    result = calculate_ingest_limits(size)
    results.append(result)

df = pd.DataFrame(results)
df['primary_hours'] = df['primary_hours'].round(2)
df['secondary_minutes'] = df['secondary_minutes'].round(2)
df['sequential_hours'] = (df['sequential_seconds'] / 3600).round(2)

print(df[['num_issues', 'total_calls', 'total_points', 'primary_hours', 'secondary_minutes', 'sequential_hours']])

   num_issues  total_calls  total_points  primary_hours  secondary_minutes  \
0         100          200           200           0.04               0.22   
1         269          538           538           0.11               0.60   
2         500         1000          1000           0.20               1.11   
3        1000         2000          2000           0.40               2.22   
4        2500         5000          5000           1.00               5.56   
5        5000        10000         10000           2.00              11.11   
6       10000        20000         20000           4.00              22.22   

   sequential_hours  
0              0.00  
1              0.01  
2              0.02  
3              0.04  
4              0.09  
5              0.19  
6              0.37  


## Analysis: Where We Hit Limits

**Primary Rate Limit:** The hourly 5,000 request limit is generous—even 5,000 issues only needs 10,000 calls (2 hours).

**Secondary Rate Limit:** This is the bottleneck. We're limited to 900 points/minute, which means:
- With 2 points per issue (comments + events)
- We can process ~450 issues per minute
- BUT we can't make requests faster than ~15 req/sec sustained

**The Problem:** Our script makes requests as fast as possible, hitting 100+ concurrent requests instantly, triggering secondary rate limits.

In [4]:
# What happens when we hit secondary limits?
import time

# Scenario: 2500 issue repo (like modelcontextprotocol/python-sdk)
num_issues = 2500
calls_needed = num_issues * 2  # comments + events
points_needed = calls_needed  # GET = 1 point each

# If we respect secondary limit (900 points/min)
min_time_seconds = (points_needed / SECONDARY_LIMIT_PER_MINUTE) * 60
min_time_hours = min_time_seconds / 3600
min_time_minutes = min_time_seconds / 60

print(f"Ingesting {num_issues} issues safely:")
print(f"  Minimum time needed: {min_time_hours:.1f} hours ({min_time_minutes:.0f} minutes)")
print(f"  Requests/second: {calls_needed / min_time_seconds:.2f} req/sec (safe)")
print(f"  Requests/minute: {calls_needed / min_time_minutes:.0f} req/min")
print(f"\nWith 0.5 second sleep between comments/events fetch:")
safe_sleep = 0.5  # seconds between API calls
safe_time = calls_needed * safe_sleep
safe_time_hours = safe_time / 3600
print(f"  Time needed: {safe_time_hours:.1f} hours")
print(f"  Requests/second: {calls_needed / safe_time:.2f} req/sec")

Ingesting 2500 issues safely:
  Minimum time needed: 0.1 hours (6 minutes)
  Requests/second: 15.00 req/sec (safe)
  Requests/minute: 900 req/min

With 0.5 second sleep between comments/events fetch:
  Time needed: 0.7 hours
  Requests/second: 2.00 req/sec


## Solution: Rate Limit Aware Ingestion

In [5]:
# Strategy 1: Add sleep delays
print("Strategy 1: Sleep Delays")
print("="*50)
for sleep_time in [0.1, 0.25, 0.5, 1.0]:
    total_time = 2500 * sleep_time  # 2500 issues, 2 calls each
    hours = total_time / 3600
    reqs_per_sec = 2500 / total_time
    print(f"  {sleep_time}s sleep between calls: {hours:.1f}h ({reqs_per_sec:.2f} req/sec)")

print("\nStrategy 2: Batch Processing")
print("="*50)
print("  Fetch all issues in list (1 call per 100 issues)")
print("  Then fetch comments/events in background, distributed over time")
print("  Allows immediate return of issue list while enriching async")

print("\nStrategy 3: Skip Heavy Calls")
print("="*50)
print("  Skip fetch_events() - often redundant")
print("  Keep only fetch_comments() - critical for resolution tracking")
print("  Reduces by 50%: 2500 issues x 1 call x 1 point = only 2500 points")
print(f"  Time needed: {2500 / SECONDARY_LIMIT_PER_MINUTE * 60 / 60:.1f} hours")

Strategy 1: Sleep Delays
  0.1s sleep between calls: 0.1h (10.00 req/sec)
  0.25s sleep between calls: 0.2h (4.00 req/sec)
  0.5s sleep between calls: 0.3h (2.00 req/sec)
  1.0s sleep between calls: 0.7h (1.00 req/sec)

Strategy 2: Batch Processing
  Fetch all issues in list (1 call per 100 issues)
  Then fetch comments/events in background, distributed over time
  Allows immediate return of issue list while enriching async

Strategy 3: Skip Heavy Calls
  Skip fetch_events() - often redundant
  Keep only fetch_comments() - critical for resolution tracking
  Reduces by 50%: 2500 issues x 1 call x 1 point = only 2500 points
  Time needed: 2.8 hours


## Response Headers: Your Rate Limit Status

In [6]:
# When you make an API call, GitHub returns these headers:
header_meanings = {
    'x-ratelimit-limit': 'Max requests allowed in this window (5000)',
    'x-ratelimit-remaining': 'Requests left this hour (decreases with each call)',
    'x-ratelimit-used': 'Requests made this hour',
    'x-ratelimit-reset': 'Unix timestamp when limit resets',
    'retry-after': '(only on 429/403) seconds to wait before retrying',
}

print("Response Headers You Should Monitor:\n")
for header, meaning in header_meanings.items():
    print(f"{header:25} -> {meaning}")

print("\n" + "="*70)
print("Example Response Headers:")
print("="*70)
example_headers = {
    'x-ratelimit-limit': '5000',
    'x-ratelimit-remaining': '4998',  # 2 calls made
    'x-ratelimit-used': '2',
    'x-ratelimit-reset': '1715792400',  # Unix timestamp
}
for h, v in example_headers.items():
    print(f"{h}: {v}")

Response Headers You Should Monitor:

x-ratelimit-limit         -> Max requests allowed in this window (5000)
x-ratelimit-remaining     -> Requests left this hour (decreases with each call)
x-ratelimit-used          -> Requests made this hour
x-ratelimit-reset         -> Unix timestamp when limit resets
retry-after               -> (only on 429/403) seconds to wait before retrying

Example Response Headers:
x-ratelimit-limit: 5000
x-ratelimit-remaining: 4998
x-ratelimit-used: 2
x-ratelimit-reset: 1715792400


## When You Get Rate Limited

**Error Response:** HTTP 429 or 403

**What to do:**
1. Check `retry-after` header (seconds to wait)
2. If no `retry-after`, check `x-ratelimit-reset` (Unix timestamp)
3. Wait that long, then retry
4. If it keeps failing, wait exponentially longer (1 min, 2 min, 4 min, etc.)

In [7]:
from datetime import datetime, timedelta
import time as time_module

# Example: handling a 429 response
reset_timestamp = 1715792400  # Example Unix timestamp
now = time_module.time()
wait_seconds = reset_timestamp - now

if wait_seconds > 0:
    reset_time = datetime.fromtimestamp(reset_timestamp)
    print(f"Rate limited! Limit resets at: {reset_time}")
    print(f"Wait {wait_seconds:.0f} seconds ({wait_seconds/60:.1f} minutes)")
else:
    print("Timestamp is in the past, limit should be reset")

Timestamp is in the past, limit should be reset


## Recommended Approach for Your Pipeline

**Current Issue:** Fetching comments + events per issue hits secondary limits

**Recommendation:** Add 0.5-1.0 second delays between comment/event fetches

This:
- Keeps you well under 900 points/minute
- Prevents 429 errors and bans
- Takes longer but completes reliably
- For 2500 issues: ~2-4 hours (acceptable)

In [8]:
# Recommended sleep strategy
print("Recommended Ingest Settings:")
print("="*50)
print("\nWait between issue fetches:")
print("  sleep(0.1)  # between pages of issues")
print("\nWait between comment/event fetches:")
print("  sleep(0.5)  # after fetching comments for each issue")
print("  sleep(0.5)  # after fetching events for each issue")
print("\nThis achieves:")
print(f"  - Avg ~2 req/sec (safe under 900 points/min)")
print(f"  - 2500 issues: ~2-3 hours")
print(f"  - 10000 issues: ~8-12 hours")
print(f"  - No 429 errors, no bans")

Recommended Ingest Settings:

Wait between issue fetches:
  sleep(0.1)  # between pages of issues

Wait between comment/event fetches:
  sleep(0.5)  # after fetching comments for each issue
  sleep(0.5)  # after fetching events for each issue

This achieves:
  - Avg ~2 req/sec (safe under 900 points/min)
  - 2500 issues: ~2-3 hours
  - 10000 issues: ~8-12 hours
  - No 429 errors, no bans
